# Google Drive Folder Explorer

This notebook helps you explore and document your Google Drive folder structure.

It will:
- Scan all files and folders in your Drive
- Count files and calculate sizes
- Generate a comprehensive guide showing your Drive structure
- Create a visual tree view of folders
- Show file type distributions

Perfect for understanding complex Drive folders with many subfolders and files!

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

## Step 2: Import the Drive Explorer

In [ ]:
import sys
import os

# Add the edgar-crawler directory to path if needed
edgar_path = '/content/drive/MyDrive/EDGAR_Project/edgar-crawler'
if os.path.exists(edgar_path):
    sys.path.insert(0, edgar_path)
    %cd {edgar_path}
    print(f"✓ Changed to: {edgar_path}")
else:
    print("⚠ EDGAR project path not found. Using current directory.")

from drive_explorer import DriveExplorer

## Step 3: Configure the Exploration

Customize these settings for your needs:

In [ ]:
# ===== CONFIGURATION =====

# Path to explore - NOW SET TO PROJECT FOLDER
ROOT_PATH = '/content/drive/MyDrive/EDGAR_Project'

# Or change to explore the entire Drive or other folders:
# ROOT_PATH = '/content/drive/MyDrive'  # Entire Drive
# ROOT_PATH = '/content/drive/MyDrive/My_10K_Files'

# Output file paths
OUTPUT_GUIDE = 'drive_structure_guide.txt'
OUTPUT_JSON = 'drive_structure.json'  # Set to None to skip JSON export

# Maximum depth to explore (None = unlimited, use a number like 3 to limit depth)
MAX_DEPTH = None

# Include options
INCLUDE_TREE_VIEW = True
INCLUDE_DETAILED_REPORT = True

print("Configuration set!")
print(f"Will explore: {ROOT_PATH}")
print(f"Output guide: {OUTPUT_GUIDE}")
if OUTPUT_JSON:
    print(f"Output JSON: {OUTPUT_JSON}")
if MAX_DEPTH:
    print(f"Max depth: {MAX_DEPTH} levels")

## Step 4: Quick Folder Summary (Optional)

Get a quick overview of a specific folder without full exploration:

In [ ]:
# Quick summary of the root folder (non-recursive)
explorer = DriveExplorer(ROOT_PATH)
summary = explorer.get_folder_summary()

print("Quick Summary:")
print("=" * 50)
print(f"Folder: {summary['name']}")
print(f"Path: {summary['path']}")
print(f"Files: {summary['files']}")
print(f"Subfolders: {summary['folders']}")
print(f"Size: {explorer._format_size(summary['size'])}")

if summary.get('file_types'):
    print("\nFile Types:")
    for ext, count in sorted(summary['file_types'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {ext}: {count} files")

## Step 5: Run Full Exploration and Generate Guide

This will scan all files and folders and create a comprehensive guide.

⚠️ **Warning**: This may take several minutes for large folder structures!

In [ ]:
print("Starting full exploration...")
print("This may take a while depending on the number of files and folders.")
print("="*80)

# Create explorer and generate guide
explorer = DriveExplorer(ROOT_PATH)

guide = explorer.generate_guide(
    output_path=OUTPUT_GUIDE,
    include_tree=INCLUDE_TREE_VIEW,
    include_detailed=INCLUDE_DETAILED_REPORT,
    max_depth=MAX_DEPTH
)

print("\n" + "="*80)
print("✓ EXPLORATION COMPLETE!")
print("="*80)
print(f"Total Files: {explorer.total_files:,}")
print(f"Total Folders: {explorer.total_folders:,}")
print(f"Total Size: {explorer._format_size(explorer.total_size)}")
print(f"\nGuide saved to: {OUTPUT_GUIDE}")

## Step 6: Export as JSON (Optional)

Export the structure as JSON for programmatic access:

In [ ]:
if OUTPUT_JSON:
    explorer.export_json(OUTPUT_JSON)
    print(f"✓ JSON exported to: {OUTPUT_JSON}")
else:
    print("JSON export skipped (set OUTPUT_JSON in configuration to enable)")

## Step 7: Preview the Guide

Let's preview the first part of the generated guide:

In [ ]:
# Preview first 100 lines of the guide
with open(OUTPUT_GUIDE, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    preview_lines = lines[:100]
    
print("Preview of the guide (first 100 lines):")
print("="*80)
print(''.join(preview_lines))

if len(lines) > 100:
    print(f"\n... (showing 100 of {len(lines)} total lines)")
    print(f"\nOpen {OUTPUT_GUIDE} to see the complete guide!")

## Step 8: File Type Analysis

Let's analyze what types of files are in your Drive:

In [ ]:
import matplotlib.pyplot as plt

# Get top file types
sorted_types = sorted(explorer.file_extensions.items(), key=lambda x: x[1], reverse=True)
top_types = sorted_types[:10]  # Top 10

if top_types:
    extensions = [ext for ext, _ in top_types]
    counts = [count for _, count in top_types]
    
    # Create bar chart
    plt.figure(figsize=(12, 6))
    plt.bar(extensions, counts, color='steelblue')
    plt.xlabel('File Extension')
    plt.ylabel('Number of Files')
    plt.title('Top 10 File Types in Your Drive')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.grid(axis='y', alpha=0.3)
    
    # Add count labels on bars
    for i, (ext, count) in enumerate(top_types):
        plt.text(i, count, f'{count:,}', ha='center', va='bottom')
    
    plt.savefig('drive_file_types.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Chart saved as: drive_file_types.png")
else:
    print("No files found to analyze.")

## Additional Utility Functions

### Search for Specific File Types

Find all files with a specific extension:

In [ ]:
def find_files_by_extension(root_path, extension):
    """Find all files with a specific extension."""
    from pathlib import Path
    
    if not extension.startswith('.'):
        extension = '.' + extension
    
    files = []
    root = Path(root_path)
    
    for file_path in root.rglob(f'*{extension}'):
        if file_path.is_file():
            files.append(str(file_path))
    
    return files

# Example: Find all CSV files
csv_files = find_files_by_extension(ROOT_PATH, '.csv')
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files[:20]:  # Show first 20
    print(f"  {f}")
if len(csv_files) > 20:
    print(f"  ... and {len(csv_files) - 20} more")

### Find Largest Files

Identify the largest files in your Drive:

In [ ]:
def find_largest_files(root_path, top_n=20):
    """Find the largest files in a directory."""
    from pathlib import Path
    
    files_with_sizes = []
    root = Path(root_path)
    
    for file_path in root.rglob('*'):
        if file_path.is_file():
            try:
                size = file_path.stat().st_size
                files_with_sizes.append((str(file_path), size))
            except (PermissionError, OSError):
                pass
    
    # Sort by size descending
    files_with_sizes.sort(key=lambda x: x[1], reverse=True)
    
    return files_with_sizes[:top_n]

# Find top 20 largest files
print("Finding largest files (this may take a moment)...")
largest_files = find_largest_files(ROOT_PATH, top_n=20)

print(f"\nTop {len(largest_files)} Largest Files:")
print("="*80)
for i, (file_path, size) in enumerate(largest_files, 1):
    size_str = explorer._format_size(size)
    print(f"{i:2d}. {size_str:>12} - {file_path}")

## Done!

Your Drive structure guide has been generated. You can:

1. **View the text guide**: Open `drive_structure_guide.txt`
2. **Analyze the JSON**: Load `drive_structure.json` for programmatic access
3. **Share the guide**: The text file is perfect for documentation
4. **Run again**: Re-run this notebook anytime your Drive structure changes

### Tips:
- If you have many files, set `MAX_DEPTH` to limit how deep it explores
- Use `INCLUDE_TREE_VIEW = False` to make the guide shorter
- Change `ROOT_PATH` to explore specific subfolders
- The JSON export is useful if you want to programmatically process the structure